# DSA_03 — Backend Contract Validation

**Purpose.** Confirm that the lightweight application layer can read and
validate the precomputed demo artifacts without importing the model artifacts.

In [1]:
# Import libraries
from pathlib import Path
import sys, yaml

In [2]:
# Define config paths
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG = PROJECT_ROOT / "configs" / "decision_support_app.yaml"
CONFIG

WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/DataLocal/ontario-electricity-peak-risk/configs/decision_support_app.yaml')

In [3]:
cfg = yaml.safe_load(CONFIG.read_text(encoding="utf-8"))

In [4]:
from src.ontario_peak_risk.decision_support.backend import DemoRepository

In [5]:
# Dislpay generated data
repo = DemoRepository(cfg, PROJECT_ROOT)

print("Available origins:", repo.available_origins)
print("FSAs:", repo.fsas)
print("Threshold:", repo.threshold)

Available origins: [Timestamp('2026-01-08 00:00:00'), Timestamp('2026-03-05 12:00:00'), Timestamp('2026-04-30 23:00:00')]
FSAs: ['L4T', 'M5R', 'M5S', 'M6G', 'M9R', 'M9W']
Threshold: 0.06


In [6]:
# Display metrics
origin = repo.available_origins[0]
data = repo.prediction_slice(origin)
metrics = repo.executive_metrics(data)

display(data.head())
metrics

,fsa,forecast_origin,target_timestamp,horizon,forecast_consumption_kwh,peak_risk_score,peak_alert
0,L4T,2026-01-08,2026-01-08 01:00:00,1,9277.822189,0.000008,False
1,L4T,2026-01-08,2026-01-08 02:00:00,2,8883.082828,0.000007,False
2,L4T,2026-01-08,2026-01-08 03:00:00,3,8682.385568,0.000009,False
3,L4T,2026-01-08,2026-01-08 04:00:00,4,8626.999043,0.000008,False
4,L4T,2026-01-08,2026-01-08 05:00:00,5,8942.451549,0.000013,False


{'expected_maximum_demand_kwh': 17808.1020441723,
 'maximum_demand_fsa': 'M9W',
 'maximum_demand_hour': Timestamp('2026-01-08 17:00:00'),
 'highest_risk_score': 0.23959587514400482,
 'highest_risk_fsa': 'M5R',
 'highest_risk_hour': Timestamp('2026-01-08 18:00:00'),
 'peak_risk_alerts': 13}

In [7]:
assert len(data) == len(repo.fsas) * repo.horizons
assert data["forecast_consumption_kwh"].gt(0).all()
assert data["peak_risk_score"].between(0, 1).all()

print("DSA_03 RESULT: PASS")

DSA_03 RESULT: PASS
